# Sierra Leone Renewable Energy UQ Analysis — V2 (Real Data)
**Author:** Kenneth Bamba | **Year:** 2026

This notebook uses real NASA POWER data for 5 Sierra Leone locations.


In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma as gammafn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import time
import os
from urllib import request
from SALib.sample import saltelli
from SALib.analyze import sobol

# Suppress known deprecation warnings from third-party libraries
warnings.filterwarnings('ignore', category=DeprecationWarning, module='SALib')

np.random.seed(42)
os.makedirs('data', exist_ok=True)
os.makedirs('figures', exist_ok=True)

LOCATIONS = {
    'Freetown': {'lat': 8.4657, 'lon': -13.2317},
    'Bo':       {'lat': 7.9647, 'lon': -11.7383},
    'Kenema':   {'lat': 7.8767, 'lon': -11.1900},
    'Makeni':   {'lat': 8.8845, 'lon': -12.0444},
    'Koidu':    {'lat': 8.6400, 'lon': -10.9700},
}

print('Setup complete.')


Setup complete.


## Section 1 — Data Acquisition (NASA POWER API)
Downloads 22 years of real daily solar and wind data for each study location.


In [2]:
def fetch_nasa_power(lat, lon, start='20010101', end='20221231'):
    params = 'ALLSKY_SFC_SW_DWN,WS10M,WS50M,T2M,RH2M'
    url = (f'https://power.larc.nasa.gov/api/temporal/daily/point'
           f'?parameters={params}&community=RE'
           f'&longitude={lon}&latitude={lat}'
           f'&start={start}&end={end}&format=JSON')
    req = request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with request.urlopen(req, timeout=60) as r:
        data = json.loads(r.read())
    p = data['properties']['parameter']
    df = pd.DataFrame(p)
    df.index = pd.to_datetime(df.index, format='%Y%m%d')
    df.index.name = 'date'
    df.columns = ['GHI','WS10M','WS50M','T2M','RH2M']
    df.replace(-999, np.nan, inplace=True)
    df.interpolate(inplace=True)
    return df

datasets = {}
for name, coords in LOCATIONS.items():
    fpath = f'data/{name.lower()}_nasa.csv'
    if os.path.exists(fpath):
        df = pd.read_csv(fpath, index_col=0, parse_dates=True)
        print(f'{name}: loaded from cache ({len(df)} days)')
    else:
        print(f'{name}: downloading...')
        df = fetch_nasa_power(coords['lat'], coords['lon'])
        df.to_csv(fpath)
        print(f'{name}: saved {len(df)} days')
        time.sleep(2)
    datasets[name] = df

print('All data ready.')


Freetown: loaded from cache (8035 days)
Bo: loaded from cache (8035 days)
Kenema: loaded from cache (8035 days)
Makeni: loaded from cache (8035 days)
Koidu: loaded from cache (8035 days)
All data ready.


## Section 2 — Exploratory Analysis


In [3]:
df = datasets['Freetown'].copy()
df['Kt'] = df['GHI'] / df['GHI'].max()  # simplified clearness proxy

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Freetown — 22-Year Daily Climate (2001-2022)', fontsize=14, fontweight='bold')

df['GHI'].resample('ME').mean().plot(ax=axes[0,0], color='orange')
axes[0,0].set_title('Monthly Mean GHI (kWh/m²/day)')
axes[0,0].set_ylabel('GHI')

df['WS50M'].resample('ME').mean().plot(ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Monthly Mean Wind Speed at 50m (m/s)')
axes[0,1].set_ylabel('WS50M')

df['T2M'].resample('ME').mean().plot(ax=axes[1,0], color='red')
axes[1,0].set_title('Monthly Mean Temperature (°C)')
axes[1,0].set_ylabel('T2M')

df['RH2M'].resample('ME').mean().plot(ax=axes[1,1], color='green')
axes[1,1].set_title('Monthly Mean Relative Humidity (%)')
axes[1,1].set_ylabel('RH2M')

plt.tight_layout()
plt.savefig('figures/01_time_series.png', dpi=150)
plt.close()
print('Figure 1 saved: figures/01_time_series.png')
print(df.describe().round(3))


Figure 1 saved: figures/01_time_series.png
            GHI     WS10M     WS50M       T2M      RH2M        Kt
count  8035.000  8035.000  8035.000  8035.000  8035.000  8035.000
mean      4.942     2.224     3.217    26.380    83.115     0.689
std       1.235     0.619     0.900     1.164     7.412     0.172
min       0.404     0.700     0.880    21.100    43.590     0.056
25%       4.247     1.790     2.590    25.540    78.150     0.592
50%       5.267     2.150     3.090    26.320    84.350     0.734
75%       5.820     2.590     3.730    27.180    89.170     0.811
max       7.174     5.870     8.290    30.000    95.240     1.000


In [4]:
df['month'] = df.index.month
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Freetown — Monthly Resource Distribution (2001-2022)', fontsize=13, fontweight='bold')

sns.boxplot(data=df, x='month', y='GHI', ax=axes[0], color='orange', linewidth=0.8)
axes[0].set_title('GHI (kWh/m\u00b2/day)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('GHI (kWh/m\u00b2/day)')

sns.boxplot(data=df, x='month', y='WS50M', ax=axes[1], color='steelblue', linewidth=0.8)
axes[1].set_title('Wind Speed 50m (m/s)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Wind Speed (m/s)')

plt.tight_layout()
plt.savefig('figures/02_seasonal_boxplots.png', dpi=150)
plt.close()
print('Figure 2 saved: figures/02_seasonal_boxplots.png')


Figure 2 saved: figures/02_seasonal_boxplots.png


## Section 3 — Solar Resource UQ (Beta Distribution)


In [5]:
# Clearness index: Kt = GHI / extraterrestrial irradiance (approx)
# Simple proxy: normalise GHI to [0,1] range
ghi = df['GHI'].dropna().values
Kt = ghi / ghi.max()
Kt = Kt[(Kt > 0.01) & (Kt < 0.99)]  # keep valid range for Beta

alpha_mle, beta_mle, loc, scale = stats.beta.fit(Kt, floc=0, fscale=1)
ks_stat, ks_p = stats.kstest(Kt, 'beta', args=(alpha_mle, beta_mle))

print(f'Beta MLE fit: alpha={alpha_mle:.3f}, beta={beta_mle:.3f}')
print(f'KS test: statistic={ks_stat:.4f}, p-value={ks_p:.4f}')
print(f'Fit is {"GOOD" if ks_p > 0.05 else "POOR"} (p-value={ks_p:.4f}, threshold=0.05)')

fig, ax = plt.subplots(figsize=(9, 5))
x = np.linspace(0, 1, 200)
ax.hist(Kt, bins=50, density=True, alpha=0.5, color='orange', label='Observed Kt')
ax.plot(x, stats.beta.pdf(x, alpha_mle, beta_mle), 'r-', lw=2,
        label=f'Beta(α={alpha_mle:.2f}, β={beta_mle:.2f})')
ax.set_xlabel('Clearness Index (Kt)')
ax.set_ylabel('Density')
ax.set_title('Freetown — Beta Distribution Fit to Clearness Index')
ax.legend()
plt.tight_layout()
plt.savefig('figures/03_beta_fit_solar.png', dpi=150)
plt.close()
print('Figure 3 saved.')


Beta MLE fit: alpha=4.662, beta=2.156
KS test: statistic=0.0762, p-value=0.0000
Fit is POOR (p-value=0.0000, threshold=0.05)
Figure 3 saved.


## Section 4 — Wind Resource UQ (Weibull Distribution)


In [6]:
ws = df['WS50M'].dropna().values
ws = ws[ws > 0]

# Method 1: MLE
shape_mle, loc_mle, scale_mle = stats.weibull_min.fit(ws, floc=0)
# Method 2: Method of Moments
mean_ws, std_ws = ws.mean(), ws.std()
# Weibull MOM approximation
k_mom = (std_ws / mean_ws) ** -1.086
c_mom = mean_ws / gammafn(1 + 1/k_mom)
# Method 3: Energy Pattern Factor
epf = np.mean(ws**3) / np.mean(ws)**3
k_epf = 1 + 3.69 / epf**2
c_epf = mean_ws / gammafn(1 + 1/k_epf)
# Method 4: Empirical (Justus)
k_emp = (std_ws / mean_ws) ** -1.086
c_emp = mean_ws / (0.568 + 0.434/k_emp)**(1/k_emp)

print('Weibull parameter estimates:')
print(f'  MLE:      k={shape_mle:.3f}, c={scale_mle:.3f}')
print(f'  MOM:      k={k_mom:.3f}, c={c_mom:.3f}')
print(f'  EPF:      k={k_epf:.3f}, c={c_epf:.3f}')
print(f'  Empirical:k={k_emp:.3f}, c={c_emp:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.linspace(0, ws.max(), 300)

axes[0].hist(ws, bins=50, density=True, alpha=0.4, color='steelblue', label='Observed')
for k, c, label in [(shape_mle, scale_mle,'MLE'),(k_mom,c_mom,'MOM'),
                     (k_epf,c_epf,'EPF'),(k_emp,c_emp,'Empirical')]:
    axes[0].plot(x, stats.weibull_min.pdf(x, k, 0, c), lw=1.5, label=f'{label} k={k:.2f}')
axes[0].set_xlabel('Wind Speed (m/s)')
axes[0].set_title('Weibull PDF — 4 Methods')
axes[0].legend(fontsize=8)

# Q-Q plot for MLE
(osm, osr), (slope, intercept, r) = stats.probplot(ws, dist=stats.weibull_min,
                                                     sparams=(shape_mle,0,scale_mle))
axes[1].scatter(osm, osr, s=2, alpha=0.3, color='steelblue')
axes[1].plot([osm.min(),osm.max()],[osm.min(),osm.max()],'r--', label=f'R²={r**2:.4f}')
axes[1].set_xlabel('Theoretical Quantiles')
axes[1].set_ylabel('Sample Quantiles')
axes[1].set_title('Q-Q Plot (MLE Weibull)')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/04_weibull_fit_wind.png', dpi=150)
plt.close()
print('Figure 4 saved.')


Weibull parameter estimates:
  MLE:      k=3.701, c=3.554
  MOM:      k=3.990, c=3.550
  EPF:      k=3.360, c=3.583
  Empirical:k=3.990, c=3.548


Figure 4 saved.


## Section 5 — Monte Carlo Energy Yield Simulation


In [7]:
N = 10000
PV_KW = 500
WIND_KW = 200

# Uncertainty parameters (based on literature)
ghi_mean  = df['GHI'].mean() * 365  # kWh/m2/yr
ghi_sigma = ghi_mean * 0.08          # 8% reanalysis bias
ws_mean   = df['WS50M'].mean()
ws_sigma  = ws_mean * 0.10           # 10% wind speed uncertainty

# Monte Carlo draws
ghi_samples = np.random.normal(ghi_mean, ghi_sigma, N)
PR_samples  = np.random.normal(0.75, 0.05, N)    # Performance Ratio
cf_wind     = np.random.normal(0.22, 0.04, N)    # Wind capacity factor

# Energy yield (MWh/yr)
PV_yield   = ghi_samples * PR_samples * PV_KW / 1000
Wind_yield = cf_wind * WIND_KW * 8760 / 1000
Total      = PV_yield + Wind_yield

# Exceedance probabilities
p_values = [50, 75, 90]
for p in p_values:
    print(f'P{p} Total AEP: {np.percentile(Total, 100-p):.1f} MWh/yr')

print(f'Mean: {Total.mean():.1f} MWh/yr')
print(f'Std:  {Total.std():.1f} MWh/yr ({Total.std()/Total.mean()*100:.1f}%)')
print(f'P90/P50 ratio: {np.percentile(Total,10)/np.percentile(Total,50):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(Total, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
for p, c in zip(p_values, ['green','orange','red']):
    v = np.percentile(Total, 100-p)
    axes[0].axvline(v, color=c, lw=2, label=f'P{p}={v:.0f} MWh')
axes[0].set_xlabel('Annual Energy Production (MWh/yr)')
axes[0].set_title(f'Monte Carlo AEP Distribution (N={N:,})')
axes[0].legend()

sorted_t = np.sort(Total)[::-1]
exc = np.arange(1, N+1) / N * 100
axes[1].plot(exc, sorted_t, color='navy')
axes[1].set_xlabel('Exceedance Probability (%)')
axes[1].set_ylabel('AEP (MWh/yr)')
axes[1].set_title('P-Exceedance Curve')
axes[1].invert_xaxis()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/05_monte_carlo_aep.png', dpi=150)
plt.close()
print('Figure 5 saved.')


P50 Total AEP: 1059.7 MWh/yr
P75 Total AEP: 992.9 MWh/yr
P90 Total AEP: 933.8 MWh/yr
Mean: 1061.4 MWh/yr
Std:  100.8 MWh/yr (9.5%)
P90/P50 ratio: 0.881


Figure 5 saved.


## Section 6 — Global Sensitivity Analysis (Sobol)


In [8]:
problem = {
    'num_vars': 4,
    'names': ['GHI_bias','PV_perf_ratio','Wind_cf','PV_capacity_kW'],
    'bounds': [[ghi_mean*0.84, ghi_mean*1.16],
               [0.65, 0.85],
               [0.14, 0.30],
               [400, 600]]
}

param_values = saltelli.sample(problem, 512, calc_second_order=False)

def aep_model(X):
    ghi_b, pr, cf_w, pv_kw = X[:,0],X[:,1],X[:,2],X[:,3]
    pv  = ghi_b * pr * pv_kw / 1000
    wnd = cf_w * WIND_KW * 8760 / 1000
    return pv + wnd

Y = aep_model(param_values)
Si = sobol.analyze(problem, Y, calc_second_order=False, print_to_console=False)

names = problem['names']
S1 = Si['S1']
ST = Si['ST']

print('Sobol Sensitivity Indices:')
for n, s1, st in zip(names, S1, ST):
    print(f'  {n:25s}  S1={s1:.3f}  ST={st:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(names))
ax.barh([n.replace('_',' ') for n in names], ST, alpha=0.8, color='navy', label='Total effect (ST)')
ax.barh([n.replace('_',' ') for n in names], S1, alpha=0.8, color='steelblue', label='First order (S1)')
ax.set_xlabel('Sobol Index')
ax.set_title('Global Sensitivity Analysis — AEP Variance Decomposition')
ax.legend()
ax.axvline(0, color='black', lw=0.5)
plt.tight_layout()
plt.savefig('figures/06_sobol_sensitivity.png', dpi=150)
plt.close()
print('Figure 6 saved.')


Sobol Sensitivity Indices:
  GHI_bias                   S1=0.202  ST=0.206
  PV_perf_ratio              S1=0.140  ST=0.144
  Wind_cf                    S1=0.338  ST=0.338
  PV_capacity_kW             S1=0.314  ST=0.319
Figure 6 saved.


/tmp/ipykernel_942084/3831151292.py:10: DeprecationWarning: `salib.sample.saltelli` will be removed in SALib 1.5.1 Please use `salib.sample.sobol`
  param_values = saltelli.sample(problem, 512, calc_second_order=False)


## Section 7 — Multi-Location Comparison


In [9]:
summary = []
for name, df_loc in datasets.items():
    annual_ghi  = df_loc['GHI'].resample('YE').sum()
    mean_ghi    = annual_ghi.mean()
    cv_ghi      = annual_ghi.std() / annual_ghi.mean() * 100
    mean_ws50   = df_loc['WS50M'].mean()
    k, loc_, c  = stats.weibull_min.fit(df_loc['WS50M'].dropna().values[df_loc['WS50M'].dropna().values > 0], floc=0)
    cf          = float(stats.weibull_min.mean(k, loc_, c)) / df_loc['WS50M'].mean() * 0.22  # rough proxy
    summary.append({'Site': name, 'Annual_GHI_kWh': round(mean_ghi,1),
                    'CV_GHI_%': round(cv_ghi,2), 'Mean_WS50_ms': round(mean_ws50,2),
                    'Weibull_k': round(k,3), 'Weibull_c': round(c,3)})

results = pd.DataFrame(summary).set_index('Site')
results.to_csv('data/multi_location_summary.csv')
print(results.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Sierra Leone — 5-Location Resource Comparison', fontsize=13, fontweight='bold')

results['Annual_GHI_kWh'].plot.bar(ax=axes[0], color='orange', alpha=0.8, rot=30)
axes[0].set_title('Mean Annual GHI (kWh/m²/yr)')
axes[0].set_ylabel('kWh/m²/yr')

results['CV_GHI_%'].plot.bar(ax=axes[1], color='steelblue', alpha=0.8, rot=30)
axes[1].set_title('Inter-annual GHI Variability (CV %)')
axes[1].set_ylabel('CV (%)')

results['Mean_WS50_ms'].plot.bar(ax=axes[2], color='green', alpha=0.8, rot=30)
axes[2].set_title('Mean Wind Speed at 50m (m/s)')
axes[2].set_ylabel('m/s')

plt.tight_layout()
plt.savefig('figures/07_multi_location.png', dpi=150)
plt.close()
print('Figure 7 saved.')


          Annual_GHI_kWh  CV_GHI_%  Mean_WS50_ms  Weibull_k  Weibull_c
Site                                                                  
Freetown          1804.8      1.32          3.22      3.701      3.554
Bo                1685.8      1.51          3.44      3.629      3.804
Kenema            1685.8      1.51          3.74      3.603      4.140
Makeni            1775.0      1.38          3.78      3.116      4.225
Koidu             1833.0      1.36          3.84      3.356      4.270


Figure 7 saved.


## Summary


In [10]:
print('='*60)
print('ANALYSIS COMPLETE — All outputs based on REAL NASA POWER data')
print('='*60)
print()
print('Figures saved to: figures/')
for f in sorted(os.listdir('figures')):
    print(f'  {f}')
print()
print('Data files saved to: data/')
for f in sorted(os.listdir('data')):
    print(f'  {f}')


ANALYSIS COMPLETE — All outputs based on REAL NASA POWER data

Figures saved to: figures/
  01_time_series.png
  02_seasonal_boxplots.png
  03_beta_fit_solar.png
  04_weibull_fit_wind.png
  05_monte_carlo_aep.png
  06_sobol_sensitivity.png
  07_multi_location.png

Data files saved to: data/
  bo_nasa.csv
  freetown_nasa.csv
  kenema_nasa.csv
  koidu_nasa.csv
  makeni_nasa.csv
  multi_location_summary.csv
